# Fraud/Anomaly Detection with Drift Monitoring

**Scenario:** A fintech transaction stream changes over time. Build a supervised fraud detector and an unsupervised Isolation Forest, compare them with Precision-Recall focused metrics, choose a cost-sensitive threshold, and monitor drift between a reference window and a later simulated window.

**Imbalance strategy:** balanced class weighting instead of naive oversampling.

**Environment:** Windows + VS Code + Python 3.13 + venv. No Anaconda, Docker, SQL or GPU required.

In [ ]:
from pathlib import Path
import json
import pandas as pd

from src.data_utils import load_data, basic_summary, split_data
from src.models import FraudModels
from src.metrics_utils import classification_metrics, find_cost_sensitive_threshold

DATA = Path('../data/demo_creditcard.csv')
if not DATA.exists():
    DATA = Path('data/demo_creditcard.csv')

df = load_data(DATA)
print(basic_summary(df))

## 1. Supervised model

The supervised model uses known fraud labels. Balanced class weights increase the loss contribution of the rare fraud class. This is preferable to simply duplicating minority examples for this small reproducible project.

In [ ]:
X_train, X_test, y_train, y_test = split_data(df)
models = FraudModels()
models.fit_supervised(X_train, y_train)

train_scores = models.supervised_scores(X_train)
test_scores = models.supervised_scores(X_test)
threshold, train_cost = find_cost_sensitive_threshold(
    y_train, train_scores, false_negative_cost=10, false_positive_cost=1
)
supervised = classification_metrics(y_test, test_scores, threshold)
supervised

## 2. Unsupervised anomaly detector

Isolation Forest is fitted without using the `Class` labels. Higher anomaly scores indicate transactions that are easier to isolate.

In [ ]:
models.fit_isolation_forest(X_train)
anomaly_scores = models.anomaly_scores(X_test)
isolation = classification_metrics(y_test, anomaly_scores, threshold=0.5)
isolation

## 3. Cost-sensitive threshold

A missed fraud transaction is assigned a cost of 10 while blocking a legitimate transaction costs 1. Therefore the threshold is selected by minimizing:

`Expected Cost = 10 × FN + 1 × FP`

This makes the decision rule business-aware rather than assuming 0.5 is automatically correct.

In [ ]:
print('Chosen threshold:', threshold)
print('Training expected cost:', train_cost)
print('Test expected cost:', supervised['fn']*10 + supervised['fp']*1)

## 4. Drift monitoring

PSI, KS statistics, and model-confidence distribution shift are used to compare the reference window with a later window. A PSI of 0.25 or more is treated as high drift. A significant confidence shift also triggers a retraining recommendation.

In [ ]:
from src.drift import feature_drift_report, confidence_drift_report, overall_drift_decision

later = df.copy()
if 'Amount' in later.columns:
    later['Amount'] = later['Amount'] * 1.35

features = [c for c in df.columns if c != 'Class']
feature_report = feature_drift_report(df, later, features)
confidence_report = confidence_drift_report(
    models.supervised_scores(df[features]),
    models.supervised_scores(later[features])
)
decision = overall_drift_decision(feature_report, confidence_report)

print(feature_report.head())
print(confidence_report)
print(decision)

## Conclusion

The supervised model uses labels and class weighting, while Isolation Forest works without labels. Precision-Recall metrics are emphasized because the fraud class is rare. The monitoring layer checks both input distribution drift and model-confidence drift, and a threshold crossing produces a retraining recommendation.